# HS4002 Week 11

## Panel Data Models

Panel data follows the **same units** (people, players, countries) across multiple time points, allowing us to control for unobserved unit-level heterogeneity.

In Python, the `linearmodels` library provides panel regression — equivalent to R's `plm` package.

We'll use NBA player data (2013–2017) to ask: **how does a player's age affect their scoring performance?**

In [ ]:
# !pip install pandas numpy plotnine linearmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

## Setup: Load and Prepare the NBA Dataset

In [ ]:
# Data source: https://www.kaggle.com/datasets/drgilermo/nba-players-stats
# NBA player statistics 1950-2017 from basketball-reference.com
raw_df = pd.read_csv("bball.csv").iloc[:, 1:]  # drop the unnamed index column

df = (
	raw_df[raw_df["Year"] >= 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"])
	.reset_index()
)
print(df.shape)
df.head()

## Setting up the Panel Index

`linearmodels` requires a **MultiIndex** of (entity, time). Equivalent to the `index=c('Player','Year')` argument in R's `plm()`.

In [ ]:
# Set the panel index — required by linearmodels
df_panel = df.set_index(['Player', 'Year'])

# A Tutorial for Panel Models

## Pooled OLS

Equivalent to a regular `lm()` in R — ignores the panel structure.

In [ ]:
pool_model = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=df_panel)
pool_result = pool_model.fit()
print(pool_result.summary)

## Fixed Effects (Within) Model

Adds `EntityEffects` to control for time-invariant player characteristics. Equivalent to `plm(..., model='within')` in R.

In [ ]:
fe_model = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=df_panel)
fe_result = fe_model.fit()
print(fe_result.summary)

Interpret the coefficients. How does the fixed-effects estimate differ from the pooled OLS estimate?

## Random Effects Model

Equivalent to `plm(..., model='random')` in R.

In [ ]:
re_model = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=df_panel)
re_result = re_model.fit()
print(re_result.summary)

## Model Diagnostics

### Lagrange Multiplier Test

Tests whether individual effects are present (i.e., whether FE/RE is needed over pooled OLS). Equivalent to R's `plmtest(fe_model, effect='individual')`.

In [ ]:
# Breusch-Pagan LM test for random effects
# Under H0 (no individual effects), we prefer pooled OLS.
# Equivalent to R's plmtest(..., effect='individual', type='bp').
from scipy.stats import chi2

resid = pool_result.resids
e_i   = resid.groupby(level=0).sum()
ssr   = float((resid ** 2).sum())
A1    = float((e_i ** 2).sum()) / ssr - 1
T_i   = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects:')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

### Hausman Test

Tests whether RE estimates are consistent (i.e., whether FE or RE is preferred). Under H0, RE is consistent and more efficient. Equivalent to R's `phtest(fe_model, re_model)`.

In [ ]:
# Hausman test: compare FE and RE estimates
# Under H0 (RE consistent), the difference in coefficients should be small
from scipy.stats import chi2

common_params = fe_result.params.index.intersection(re_result.params.index)

beta_diff = fe_result.params[common_params] - re_result.params[common_params]
cov_diff  = (
	fe_result.cov.loc[common_params, common_params] -
	re_result.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val  = chi2.sf(h_stat, df=len(common_params))
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

## Comparing the Models

As in earlier weeks, `stargazer` collects the fitted models into one regression table. Reading pooled OLS, fixed effects, and random effects side by side is easier than scrolling through three summaries. Equivalent to R's `stargazer(lm1, fe_glm1, re_glm1)`.

In [ ]:
from stargazer.stargazer import Stargazer

sg = Stargazer([pool_result, fe_result, re_result])
sg.title('Points per Game and Age')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Points per game')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg.covariate_order(['age', 'I(age ** 2)'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
})

# Ending the cell on the object itself renders the table in the notebook
sg

# Independent Analysis: Has Sports Science Extended Career Peaks?

Compare players from 2008–2012 with players from 2013–2017. Does age affect scoring differently across the two eras?

In [ ]:
# (1) create two data sets, one for each era.

In [ ]:
# (2) fit models for each of the two data sets

In [ ]:
# (3) run diagnostics, compare, and interpret for 2008-2012.

In [ ]:
# (4) run diagnostics, compare, and interpret for 2013-2017.

In [ ]:
# (5) what differences do you find in the age-scoring relationship across the two eras?